# Practical 1: Multi-Format Data Parsing, Binary Files, Regular Expressions & Relational CRUD

**Name:** Vedant Vinod Rakshe

**Aim:** To write programs that parse text, CSV, HTML, XML and JSON documents, extract the relevant data and check it for anomalies and missing values; to read and write binary files; to search, split and replace strings using regular expressions; and to design a small relational database and perform CRUD operations using SQL.

## Task 1: Setup - Create Sample Input Files

In [1]:
# Name: Vedant Vinod Rakshe
# Practical 1 - Setup: create the sample input files used in all tasks
print("Name: Vedant Vinod Rakshe")
import json

with open("sample.txt", "w") as f:
    f.write("# Student records (key:value format)\n")
    f.write("id: 1, name: Amit, age: 21, marks: 85, city: Mumbai\n")
    f.write("id: 2, name: Riya, age: 22, marks: 90, city: Pune\n")
    f.write("id: 3, name: Neha, age: None, marks: 78, city: Mumbai\n")
    f.write("\n")
    f.write("id: 4, name: Karan, age: 23, marks: -5, city: Delhi\n")
    f.write("id: 5, name: Sneha, age: 200, marks: 88, city: Pune\n")
    f.write("id: 2, name: Riya, age: 22, marks: 90, city: Pune\n")
    f.write("id: 6, name: Rahul, age: 24, marks: None, city: Nashik\n")

with open("sample.csv", "w") as f:
    f.write("id,name,age,marks,city\n1,Amit,21,85,Mumbai\n2,Riya,22,90,Pune\n")
    f.write("3,Neha,,78,Mumbai\n4,Karan,23,-5,Delhi\n5,Sneha,200,88,Pune\n")
    f.write("2,Riya,22,90,Pune\n6,Rahul,24,,Nashik\n")

with open("sample.html", "w") as f:
    f.write("""<html><body><h1>Student Result Page</h1>
<a href="https://example.com/results">Results</a>
<table id="students"><tr><th>id</th><th>name</th><th>age</th><th>marks</th><th>city</th></tr>
<tr><td>1</td><td>Amit</td><td>21</td><td>85</td><td>Mumbai</td></tr>
<tr><td>2</td><td>Riya</td><td>22</td><td>90</td><td>Pune</td></tr>
<tr><td>3</td><td>Neha</td><td></td><td>78</td><td>Mumbai</td></tr>
<tr><td>4</td><td>Karan</td><td>23</td><td>-5</td><td>Delhi</td></tr>
<tr><td>5</td><td>Sneha</td><td>200</td><td>88</td><td>Pune</td></tr>
<tr><td>2</td><td>Riya</td><td>22</td><td>90</td><td>Pune</td></tr>
<tr><td>6</td><td>Rahul</td><td>24</td><td></td><td>Nashik</td></tr>
</table></body></html>""")

with open("sample.xml", "w") as f:
    f.write("""<students>
<student id="1"><name>Amit</name><age>21</age><marks>85</marks><city>Mumbai</city></student>
<student id="2"><name>Riya</name><age>22</age><marks>90</marks><city>Pune</city></student>
<student id="3"><name>Neha</name><marks>78</marks><city>Mumbai</city></student>
<student id="4"><name>Karan</name><age>23</age><marks>-5</marks><city>Delhi</city></student>
<student id="5"><name>Sneha</name><age>200</age><marks>88</marks><city>Pune</city></student>
<student id="2"><name>Riya</name><age>22</age><marks>90</marks><city>Pune</city></student>
<student id="6"><name>Rahul</name><age>24</age><city>Nashik</city></student>
</students>""")

students = [
    {"id": 1, "name": "Amit", "age": 21, "marks": 85, "address": {"city": "Mumbai"}},
    {"id": 2, "name": "Riya", "age": 22, "marks": 90, "address": {"city": "Pune"}},
    {"id": 3, "name": "Neha", "age": None, "marks": 78, "address": {"city": "Mumbai"}},
    {"id": 4, "name": "Karan", "age": 23, "marks": -5, "address": {"city": "Delhi"}},
    {"id": 5, "name": "Sneha", "age": 200, "marks": 88, "address": {"city": "Pune"}},
    {"id": 2, "name": "Riya", "age": 22, "marks": 90, "address": {"city": "Pune"}},
    {"id": 6, "name": "Rahul", "age": 24, "marks": None, "address": {"city": "Nashik"}},
]
with open("sample.json", "w") as f:
    json.dump(students, f, indent=2)
print("Sample files created: sample.txt, sample.csv, sample.html, sample.xml, sample.json")

Name: Vedant Vinod Rakshe
Sample files created: sample.txt, sample.csv, sample.html, sample.xml, sample.json


## Task 2: Data Quality Inspection Helper

The same set of student records is stored in five formats. Each record set contains deliberate problems (a missing age, a negative mark, an impossible age of 200, a duplicate row and a missing mark). One helper function reports missing values, duplicates and out-of-range values for every format.

In [ ]:
# Name: Vedant Vinod Rakshe
import pandas as pd

def inspect_data(df, title, rules=None):
    """Prints a data-quality report: missing values, duplicates, anomalies."""
    rules = rules or {"age": (1, 100), "marks": (0, 100)}   # valid ranges
    for col in ["id", "age", "marks"]:                      # fix data types
        df[col] = pd.to_numeric(df[col], errors="coerce")
    print(f"===== Data Quality Report: {title} =====")
    print("Shape:", df.shape)
    print("\nMissing values per column:")
    print(df.isnull().sum().to_string())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nDomain anomalies (values outside valid range):")
    found = False
    for col, (lo, hi) in rules.items():
        for _, r in df[(df[col] < lo) | (df[col] > hi)].iterrows():
            print(f"  id={int(r['id'])}: {col}={r[col]:g} is outside [{lo}, {hi}]")
            found = True
    if not found:
        print("  None")
    return df

## Task 3: Parse a Text File and Inspect Data Quality

In [2]:
print("Name: Vedant Vinod Rakshe")
def parse_text_file(filepath):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):      # skip blank/comment lines
                continue
            record = {}
            for part in line.split(","):
                if ":" in part:
                    k, v = part.split(":", 1)
                    record[k.strip()] = None if v.strip() == "None" else v.strip()
            if record:
                records.append(record)
    return pd.DataFrame(records)

df_txt = parse_text_file("sample.txt")
print(df_txt, end="\n\n")
df_txt = inspect_data(df_txt, "TXT Document")

Name: Vedant Vinod Rakshe
  id   name  age marks    city
0  1   Amit   21    85  Mumbai
1  2   Riya   22    90    Pune
2  3   Neha  NaN    78  Mumbai
3  4  Karan   23    -5   Delhi
4  5  Sneha  200    88    Pune
5  2   Riya   22    90    Pune
6  6  Rahul   24   NaN  Nashik

===== Data Quality Report: TXT Document =====
Shape: (7, 5)

Missing values per column:
id       0
name     0
age      1
marks    1
city     0

Duplicate rows: 1

Domain anomalies (values outside valid range):
  id=5: age=200 is outside [1, 100]
  id=4: marks=-5 is outside [0, 100]


## Task 4: Parse a CSV File and Identify Domain Anomalies

In [3]:
print("Name: Vedant Vinod Rakshe")
def parse_csv_file(filepath):
    return pd.read_csv(filepath)

df_csv = parse_csv_file("sample.csv")
print(df_csv, end="\n\n")
df_csv = inspect_data(df_csv, "CSV Document")

Name: Vedant Vinod Rakshe
   id   name    age  marks    city
0   1   Amit   21.0   85.0  Mumbai
1   2   Riya   22.0   90.0    Pune
2   3   Neha    NaN   78.0  Mumbai
3   4  Karan   23.0   -5.0   Delhi
4   5  Sneha  200.0   88.0    Pune
5   2   Riya   22.0   90.0    Pune
6   6  Rahul   24.0    NaN  Nashik

===== Data Quality Report: CSV Document =====
Shape: (7, 5)

Missing values per column:
id       0
name     0
age      1
marks    1
city     0

Duplicate rows: 1

Domain anomalies (values outside valid range):
  id=5: age=200 is outside [1, 100]
  id=4: marks=-5 is outside [0, 100]


## Task 5: Parse an HTML Document

In [4]:
print("Name: Vedant Vinod Rakshe")
from bs4 import BeautifulSoup

def parse_html_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    print("Page heading :", soup.h1.text)
    print("Links found  :", [a["href"] for a in soup.find_all("a")], end="\n\n")
    table = soup.find("table", id="students")
    headers = [th.text for th in table.find_all("th")]
    rows = []
    for tr in table.find_all("tr")[1:]:
        cells = [td.text.strip() or None for td in tr.find_all("td")]
        rows.append(dict(zip(headers, cells)))
    return pd.DataFrame(rows)

df_html = parse_html_file("sample.html")
print(df_html, end="\n\n")
df_html = inspect_data(df_html, "HTML Document")

Name: Vedant Vinod Rakshe
Page heading : Student Result Page
Links found  : ['https://example.com/results']

  id   name  age marks    city
0  1   Amit   21    85  Mumbai
1  2   Riya   22    90    Pune
2  3   Neha  NaN    78  Mumbai
3  4  Karan   23    -5   Delhi
4  5  Sneha  200    88    Pune
5  2   Riya   22    90    Pune
6  6  Rahul   24   NaN  Nashik

===== Data Quality Report: HTML Document =====
Shape: (7, 5)

Missing values per column:
id       0
name     0
age      1
marks    1
city     0

Duplicate rows: 1

Domain anomalies (values outside valid range):
  id=5: age=200 is outside [1, 100]
  id=4: marks=-5 is outside [0, 100]


## Task 6: Parse an XML Document

In [5]:
print("Name: Vedant Vinod Rakshe")
import xml.etree.ElementTree as ET

def parse_xml_file(filepath):
    root = ET.parse(filepath).getroot()
    rows = []
    for s in root.findall("student"):
        rows.append({
            "id": s.get("id"),
            "name": s.findtext("name"),
            "age": s.findtext("age"),        # None if the tag is missing
            "marks": s.findtext("marks"),
            "city": s.findtext("city"),
        })
    return pd.DataFrame(rows)

df_xml = parse_xml_file("sample.xml")
print(df_xml, end="\n\n")
df_xml = inspect_data(df_xml, "XML Document")

Name: Vedant Vinod Rakshe
  id   name  age marks    city
0  1   Amit   21    85  Mumbai
1  2   Riya   22    90    Pune
2  3   Neha  NaN    78  Mumbai
3  4  Karan   23    -5   Delhi
4  5  Sneha  200    88    Pune
5  2   Riya   22    90    Pune
6  6  Rahul   24   NaN  Nashik

===== Data Quality Report: XML Document =====
Shape: (7, 5)

Missing values per column:
id       0
name     0
age      1
marks    1
city     0

Duplicate rows: 1

Domain anomalies (values outside valid range):
  id=5: age=200 is outside [1, 100]
  id=4: marks=-5 is outside [0, 100]


## Task 7: Parse a JSON Document

In [6]:
print("Name: Vedant Vinod Rakshe")
import json

def parse_json_file(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)
    return pd.json_normalize(data)            # flattens nested address.city

df_json = parse_json_file("sample.json").rename(columns={"address.city": "city"})
print(df_json, end="\n\n")
df_json = inspect_data(df_json, "JSON Document")

Name: Vedant Vinod Rakshe
   id   name    age  marks    city
0   1   Amit   21.0   85.0  Mumbai
1   2   Riya   22.0   90.0    Pune
2   3   Neha    NaN   78.0  Mumbai
3   4  Karan   23.0   -5.0   Delhi
4   5  Sneha  200.0   88.0    Pune
5   2   Riya   22.0   90.0    Pune
6   6  Rahul   24.0    NaN  Nashik

===== Data Quality Report: JSON Document =====
Shape: (7, 5)

Missing values per column:
id       0
name     0
age      1
marks    1
city     0

Duplicate rows: 1

Domain anomalies (values outside valid range):
  id=5: age=200 is outside [1, 100]
  id=4: marks=-5 is outside [0, 100]


## Task 8: Reading and Writing Binary Files

In [7]:
print("Name: Vedant Vinod Rakshe")
import struct, os, hashlib

# ---- WRITE a binary file: each record = int id, 10-byte name, float marks
records = [(1, "Amit", 85.5), (2, "Riya", 90.0), (3, "Neha", 78.25)]
fmt = "<i10sf"
size = struct.calcsize(fmt)
with open("students.bin", "wb") as f:
    for rid, name, marks in records:
        f.write(struct.pack(fmt, rid, name.encode().ljust(10, b"\x00"), marks))
print("Record size:", size, "bytes | File size:", os.path.getsize("students.bin"), "bytes")

# ---- READ the binary file back
with open("students.bin", "rb") as f:
    print("First record (hex):", f.read(size).hex(" "))
    f.seek(0)
    while chunk := f.read(size):
        rid, name, marks = struct.unpack(fmt, chunk)
        print("Read ->", rid, name.rstrip(b"\x00").decode(), marks)

# ---- UPDATE record 2 in place using seek()
with open("students.bin", "r+b") as f:
    f.seek(size * 1)
    f.write(struct.pack(fmt, 2, b"Riya".ljust(10, b"\x00"), 95.0))
with open("students.bin", "rb") as f:
    f.seek(size * 1)
    print("Updated record 2 ->", struct.unpack(fmt, f.read(size)))

# ---- COPY the binary file in chunks and verify with an MD5 checksum
with open("students.bin", "rb") as src, open("students_copy.bin", "wb") as dst:
    while block := src.read(16):
        dst.write(block)
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
print("Original MD5:", md5("students.bin"))
print("Copy MD5    :", md5("students_copy.bin"))
print("Files identical:", md5("students.bin") == md5("students_copy.bin"))

Name: Vedant Vinod Rakshe
Record size: 18 bytes | File size: 54 bytes
First record (hex): 01 00 00 00 41 6d 69 74 00 00 00 00 00 00 00 00 ab 42
Read -> 1 Amit 85.5
Read -> 2 Riya 90.0
Read -> 3 Neha 78.25
Updated record 2 -> (2, b'Riya\x00\x00\x00\x00\x00\x00', 95.0)
Original MD5: c91f954e19701507ab1c8f6834e7dc04
Copy MD5    : c91f954e19701507ab1c8f6834e7dc04
Files identical: True


## Task 9: Searching, Splitting and Replacing Using Regular Expressions

In [8]:
print("Name: Vedant Vinod Rakshe")
import re

text = """Contact: amit.k@mail.com, riya_90@college.edu.in | Phone: 98765-43210, 91234 56789
Joined on 12-08-2023, left on 05/01/2025. Remarks: good;average,poor excellent"""

# 1. SEARCHING
print("First email :", re.search(r"[\w.]+@[\w.]+\.\w+", text).group())
print("All emails  :", re.findall(r"[\w.]+@[\w.]+\.\w+", text))
print("All phones  :", re.findall(r"\b\d{5}[- ]\d{5}\b", text))
print("All dates   :", re.findall(r"\b\d{2}[-/]\d{2}[-/]\d{4}\b", text))

# 2. SPLITTING (on ; , or whitespace)
remarks = text.split("Remarks: ")[1]
print("Split remarks:", re.split(r"[;,\s]+", remarks))

# 3. REPLACING
masked = re.sub(r"\b(\d{5})[- ](\d{5})\b", r"XXXXX-\2", text)
print("Masked phones:", re.findall(r"XXXXX-\d{5}", masked))
iso = re.sub(r"\b(\d{2})[-/](\d{2})[-/](\d{4})\b", r"\3-\2-\1", text)
print("ISO dates    :", re.findall(r"\b\d{4}-\d{2}-\d{2}\b", iso))
hidden = re.sub(r"(?<=\w)[\w.]*(?=@)", "***", text)
print("Hidden emails:", re.findall(r"\*\*\*@[\w.]+", hidden))

Name: Vedant Vinod Rakshe
First email : amit.k@mail.com
All emails  : ['amit.k@mail.com', 'riya_90@college.edu.in']
All phones  : ['98765-43210', '91234 56789']
All dates   : ['12-08-2023', '05/01/2025']
Split remarks: ['good', 'average', 'poor', 'excellent']
Masked phones: ['XXXXX-43210', 'XXXXX-56789']
ISO dates    : ['2023-08-12', '2025-01-05']
Hidden emails: ['***@mail.com', '***@college.edu.in']


## Task 10: Relational Database Design and CRUD Operations Using SQL

Application: a small College Enrolment system with three tables - Student, Course and Enrollment (a many-to-many link table using foreign keys).

In [9]:
print("Name: Vedant Vinod Rakshe")
import sqlite3

conn = sqlite3.connect("college.db")
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")
cur.executescript("""
DROP TABLE IF EXISTS Enrollment; DROP TABLE IF EXISTS Student; DROP TABLE IF EXISTS Course;
CREATE TABLE Student (
    student_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT);
CREATE TABLE Course (
    course_id INTEGER PRIMARY KEY, title TEXT NOT NULL, credits INTEGER CHECK(credits > 0));
CREATE TABLE Enrollment (
    enroll_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER REFERENCES Student(student_id) ON DELETE CASCADE,
    course_id  INTEGER REFERENCES Course(course_id),
    grade TEXT);
""")

# ---------- C : CREATE (insert rows) ----------
cur.executemany("INSERT INTO Student VALUES (?,?,?)",
    [(1, "Amit", "Mumbai"), (2, "Riya", "Pune"), (3, "Neha", "Mumbai")])
cur.executemany("INSERT INTO Course VALUES (?,?,?)",
    [(101, "Data Engineering", 4), (102, "Databases", 3), (103, "Python", 3)])
cur.executemany("INSERT INTO Enrollment (student_id, course_id, grade) VALUES (?,?,?)",
    [(1, 101, "A"), (1, 102, "B"), (2, 101, "A"), (3, 103, "B")])
conn.commit()

def show(title, sql):
    print(f"--- {title} ---")
    for row in cur.execute(sql): print(row)

# ---------- R : READ ----------
show("READ: all students", "SELECT * FROM Student")
show("READ: JOIN of enrolments", """SELECT s.name, c.title, e.grade FROM Enrollment e
    JOIN Student s ON s.student_id = e.student_id
    JOIN Course c ON c.course_id = e.course_id ORDER BY s.name""")
show("READ: students per city", "SELECT city, COUNT(*) FROM Student GROUP BY city")

# ---------- U : UPDATE ----------
cur.execute("UPDATE Student SET city = 'Nashik' WHERE student_id = 3")
cur.execute("UPDATE Enrollment SET grade = 'A' WHERE student_id = 1 AND course_id = 102")
conn.commit()
show("AFTER UPDATE: students", "SELECT * FROM Student")

# ---------- D : DELETE ----------
cur.execute("DELETE FROM Student WHERE student_id = 2")   # cascades to Enrollment
conn.commit()
show("AFTER DELETE: students", "SELECT * FROM Student")
show("AFTER DELETE: enrolments", "SELECT * FROM Enrollment")
conn.close()

Name: Vedant Vinod Rakshe
--- READ: all students ---
(1, 'Amit', 'Mumbai')
(2, 'Riya', 'Pune')
(3, 'Neha', 'Mumbai')
--- READ: JOIN of enrolments ---
('Amit', 'Data Engineering', 'A')
('Amit', 'Databases', 'B')
('Neha', 'Python', 'B')
('Riya', 'Data Engineering', 'A')
--- READ: students per city ---
('Mumbai', 2)
('Pune', 1)
--- AFTER UPDATE: students ---
(1, 'Amit', 'Mumbai')
(2, 'Riya', 'Pune')
(3, 'Neha', 'Nashik')
--- AFTER DELETE: students ---
(1, 'Amit', 'Mumbai')
(3, 'Neha', 'Nashik')
--- AFTER DELETE: enrolments ---
(1, 1, 101, 'A')
(2, 1, 102, 'A')
(4, 3, 103, 'B')


**Conclusion:** Data in five different formats was parsed into a common DataFrame, and the same inspection routine detected the missing values, duplicate record and out-of-range values in each. Binary file handling, regular-expression based text processing and SQL CRUD operations on a relational schema were also implemented successfully.